# DSA Drill — Lowest Common Ancestor of a Binary Tree (LC 236 · Medium)

> Given a binary tree and two nodes `p` and `q`, return their **lowest common ancestor** — the lowest node with both `p` and `q` in its subtree (a node counts as its own descendant).

**Examples**
1. `p=5, q=1` → LCA = 3 (the root)
2. `p=5, q=4` → LCA = 5 — *a node can be the descendant of itself*
3. `p = q = root` → LCA = root

**Protocol (20:00):** verbalize the recursion contract BEFORE coding — what can a call return, and what does a parent do with three possible answers (`None` / `p-or-q found` / `LCA found`)?

**Landmines**
1. The "both sides non-None → I am the LCA" rule — why is it correct even when `p` is an ancestor of `q`?
2. If only one subtree returns non-None, what goes upward and why?
3. Works when `p` or `q` IS the root?

One hint allowed (costs the badge).

## Intuition (plain): report upward, decide where you stand

Ask every subtree the same question: *"do you contain p or q?"* A leaf answers `None` (no) — or the node itself if it **is** `p`/`q`. A parent looks at both answers:

```text
left  = recurse(left)    # None | found-target | already-found LCA
right = recurse(right)

if left and right:       # one target on each side -> I am the LCA
    return me
return left or right     # both on one side (or I am p/q myself) -> pass it up
```

The elegance: the moment the answer exists in a subtree, it bubbles up unchanged — no counts, no paths, no parent pointers.

In [1]:
class Node:
    __slots__ = ("val", "left", "right")

    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"Node({self.val!r}, left={self.left!r}, right={self.right!r})"


def lowest_common_ancestor(root, p, q):
    """Return the LCA node of p and q in a binary tree (single pass, O(n))."""
    if root is None or root is p or root is q:
        return root

    left = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)

    if left and right:
        return root
    return left or right

In [3]:
import random

# Use the Node class defined above; do not redefine a different node class here.
random.seed(17)

def contains(node, target):
    if node is None:
        return False
    return node is target or contains(node.left, target) or contains(node.right, target)

# Fixed example tree:
#            3
#          /   \
#         5     1
#        / \   / \
#       6   2 0   8
#          / \
#         7   4
n7, n4, n6, n0, n8 = Node(7), Node(4), Node(6), Node(0), Node(8)
n2 = Node(2, n7, n4)
n5 = Node(5, n6, n2)
n1 = Node(1, n0, n8)
root = Node(3, n5, n1)

assert lowest_common_ancestor(root, n5, n1) is root
assert lowest_common_ancestor(root, n5, n4) is n5
assert lowest_common_ancestor(root, root, n5) is root
assert lowest_common_ancestor(root, n7, n4) is n2

# Random-tree property checks using the same Node class defined above
for trial in range(200):
    nodes = [Node(i) for i in range(25)]
    tree_root = nodes[0]
    attached = [tree_root]

    for i in range(1, 25):
        parent = random.choice(attached)
        slot = "left" if random.random() < 0.5 else "right"
        if getattr(parent, slot) is None:
            setattr(parent, slot, nodes[i])
            attached.append(nodes[i])

    p, q = random.choice(attached), random.choice(attached)
    lca = lowest_common_ancestor(tree_root, p, q)

    assert lca is not None and contains(lca, p) and contains(lca, q), (p.val, q.val)
    for child in (lca.left, lca.right):
        assert not (child is not None and contains(child, p) and contains(child, q)), (
            p.val,
            q.val,
            "LCA not lowest",
        )

print("fixed cases + 200 random-tree property checks pass")

fixed cases + 200 random-tree property checks pass


In [4]:
# Fixed example tree:
#            3
#          /   \
#         5     1
#        / \   / \
#       6   2 0   8
#          / \
#         7   4
n7, n4, n6, n0, n8 = Node(7), Node(4), Node(6), Node(0), Node(8)
n2 = Node(2, n7, n4)
n5 = Node(5, n6, n2)
n1 = Node(1, n0, n8)
root = Node(3, n5, n1)

assert lowest_common_ancestor(root, n5, n1) is root        # example 1
assert lowest_common_ancestor(root, n5, n4) is n5          # example 2: ancestor case
assert lowest_common_ancestor(root, root, n5) is root      # root as target
assert lowest_common_ancestor(root, n7, n4) is n2          # siblings under 2

# Fuzz on random trees with a property check: the LCA must contain both targets,
# and NO child of the LCA may contain both (deepest such node).
import random
random.seed(17)

def contains(node, target):
    if node is None:
        return False
    return node is target or contains(node.left, target) or contains(node.right, target)

for trial in range(200):
    class MNode:
        __slots__ = ('val', 'left', 'right')
        def __init__(self, val):
            self.val, self.left, self.right = val, None, None

    class MNode:
        __slots__ = ('val', 'left', 'right')
        def __init__(self, val):
            self.val, self.left, self.right = val, None, None

    nodes = [MNode(i) for i in range(25)]   # MUTABLE nodes: namedtuples reject setattr
    tree_root = nodes[0]
    attached = [tree_root]
    for i in range(1, 25):
        parent = random.choice(attached)      # pick a parent that is IN the tree
        slot = 'left' if random.random() < 0.5 else 'right'
        if getattr(parent, slot) is None:
            setattr(parent, slot, nodes[i])
            attached.append(nodes[i])
        # else: skip this node entirely (never attached; not sampled below)
    p, q = random.choice(attached), random.choice(attached)
    lca = lowest_common_ancestor(tree_root, p, q)
    assert lca is not None and contains(lca, p) and contains(lca, q), (p.val, q.val)
    for child in (lca.left, lca.right):
        assert not (child is not None and contains(child, p) and contains(child, q)), \
            (p.val, q.val, 'LCA not lowest')
print("fixed cases + 200 random-tree property checks pass")

fixed cases + 200 random-tree property checks pass


In [5]:
# Edge cases - run me, say WHY out loud
solo = Node(1)
assert lowest_common_ancestor(solo, solo, solo) is solo   # single-node tree
assert lowest_common_ancestor(None, n5, n1) is None       # empty tree
print("edge cases pass OK")

edge cases pass OK


## Post-mortem (fill in your own words)

- The recursion contract (three possible answers):
- Why "both sides non-None = me" survives the ancestor case:
- Complexity (every node visited once?):
- One-line pattern (copy into dsa/patterns.md):